# 15.7 Two-tower & neural retrieval recommenders

Two towers learn a user vector and an item vector so retrieval becomes nearest-neighbor search at scale.

This notebook scales recommendation from scoring a few candidates to retrieving from millions. Matrix factorization supplies the dot-product scoring idea, sampled softmax trains positives against negatives, and nearest-neighbor search is approximated here by brute-force NumPy top-k.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(1507)


def softmax(scores, temperature=1.0):
    values = np.asarray(scores, dtype=float) / float(temperature)
    values = values - np.max(values)
    weights = np.exp(values)
    return weights / np.sum(weights)


def normalize_rows(matrix):
    values = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return values / norms


def top_k_indices(scores, k):
    values = np.asarray(scores, dtype=float)
    order = np.argsort(-values)
    return order[:k]


def recall_at_k(scores, positives, k=3):
    values = np.asarray(scores, dtype=float)
    hits = []
    for row, pos in zip(values, positives):
        top = set(top_k_indices(row, k))
        wanted = set(np.atleast_1d(pos).astype(int))
        hits.append(len(top & wanted) / max(1, len(wanted)))
    return float(np.mean(hits))


def dcg_at_k(relevance, k=3):
    rel = np.asarray(relevance, dtype=float)[:k]
    gains = np.power(2.0, rel) - 1.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    return float(np.sum(gains / discounts))


def ndcg_at_k(scores, relevance, k=3):
    values = np.asarray(scores, dtype=float)
    rel = np.asarray(relevance, dtype=float)
    order = np.argsort(-values)[:k]
    ideal = np.argsort(-rel)[:k]
    ideal_dcg = dcg_at_k(rel[ideal], k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(rel[order], k) / ideal_dcg


def mrr_from_scores(scores, target):
    order = np.argsort(-np.asarray(scores, dtype=float))
    rank = int(np.where(order == int(target))[0][0]) + 1
    return 1.0 / rank


def make_f14_ladder(seed=1514):
    rng = np.random.default_rng(seed)
    rungs = []

    item_vectors = np.array([
        [1.0, 0.1, 0.0],
        [0.8, 0.4, 0.0],
        [0.1, 1.0, 0.2],
        [-0.2, 0.3, 0.9],
    ])
    user_vectors = np.array([
        [1.0, 0.2, 0.0],
        [0.2, 1.0, 0.2],
        [-0.1, 0.4, 1.0],
    ])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D1 tiny slate",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.array([0]), np.array([2]), np.array([3])],
        "sessions": [[0, 1, 2], [1, 2, 0], [2, 3, 1]],
        "targets": [2, 0, 1],
        "clicks": np.array([5, 8, 1]),
        "impressions": np.array([100, 100, 20]),
        "eligible": np.array([True, True, True, True]),
        "exposure": np.array([1.0, 0.8, 0.5, 0.3]),
    })

    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    item_vectors = np.column_stack([np.cos(angles), np.sin(angles), 0.35 * np.cos(2.0 * angles)])
    user_angles = np.array([0.1, 1.6, 3.0, 4.7, 5.6])
    user_vectors = np.column_stack([np.cos(user_angles), np.sin(user_angles), 0.25 * np.sin(user_angles)])
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D2 synthetic preferences",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 1, 2, 3], [2, 3, 4], [4, 5, 6], [6, 7, 0], [7, 0, 1]],
        "targets": [3, 4, 6, 0, 1],
        "clicks": np.array([12, 20, 18, 6, 4, 3, 9, 7]),
        "impressions": np.array([200, 260, 220, 150, 80, 60, 120, 140]),
        "eligible": np.ones(8, dtype=bool),
        "exposure": np.linspace(1.0, 0.35, 8),
    })

    item_vectors = rng.normal(size=(12, 4))
    item_vectors = normalize_rows(item_vectors)
    user_vectors = rng.normal(size=(7, 4))
    user_vectors = normalize_rows(user_vectors)
    relevance = user_vectors @ item_vectors.T
    relevance = relevance + rng.normal(scale=0.04, size=relevance.shape)
    exposure = rng.uniform(0.15, 1.0, size=12)
    observed = rng.uniform(size=relevance.shape) < exposure[None, :]
    sparse_relevance = np.where(observed, relevance, -0.2)
    rungs.append({
        "name": "D3 sparse noisy logs",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": sparse_relevance,
        "true_relevance": relevance,
        "positives": [np.argsort(-row)[:2] for row in relevance],
        "sessions": [[0, 2, 5, 7], [1, 2, 4], [3, 8, 9], [10, 2, 0], [6, 6, 11], [4, 5, 6], [9, 1, 3]],
        "targets": [7, 4, 9, 0, 11, 6, 3],
        "clicks": np.array([30, 16, 9, 4, 3, 2, 8, 7, 5, 4, 2, 1]),
        "impressions": np.array([600, 280, 190, 90, 45, 30, 120, 110, 70, 65, 22, 12]),
        "eligible": rng.uniform(size=12) > 0.10,
        "exposure": exposure,
    })

    genres = np.array([
        [1, 0, 0, 0, 1],
        [1, 1, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 1, 1, 0],
        [0, 0, 0, 1, 1],
        [1, 0, 0, 1, 0],
        [0, 1, 0, 0, 1],
        [0, 0, 1, 0, 1],
        [1, 0, 1, 0, 0],
        [0, 1, 0, 1, 0],
    ], dtype=float)
    item_vectors = normalize_rows(genres + 0.05 * rng.normal(size=genres.shape))
    user_vectors = normalize_rows(np.array([
        [2, 0, 0, 0, 1],
        [1, 2, 0, 0, 0],
        [0, 1, 2, 0, 0],
        [0, 0, 1, 2, 0],
        [0, 0, 0, 1, 2],
        [1, 0, 1, 0, 1],
    ], dtype=float))
    relevance = user_vectors @ item_vectors.T
    rungs.append({
        "name": "D4 MovieLens-like inline",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 5, 8], [1, 2, 6], [2, 3, 8], [3, 4, 9], [4, 7, 0], [7, 8, 6]],
        "targets": [8, 6, 3, 9, 4, 6],
        "clicks": np.array([50, 34, 25, 18, 17, 14, 9, 7, 6, 5]),
        "impressions": np.array([1000, 700, 520, 430, 320, 250, 150, 110, 90, 80]),
        "eligible": np.ones(10, dtype=bool),
        "exposure": np.array([1.0, 0.9, 0.75, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25, 0.20]),
    })

    head = normalize_rows(rng.normal(loc=0.6, scale=0.25, size=(5, 5)))
    torso = normalize_rows(rng.normal(loc=0.0, scale=0.7, size=(8, 5)))
    cold = normalize_rows(np.array([
        [1, 1, 0, 0, 1],
        [0, 1, 1, 1, 0],
        [1, 0, 1, 0, 1],
        [0, 0, 1, 1, 1],
    ], dtype=float))
    item_vectors = np.vstack([head, torso, cold])
    user_vectors = normalize_rows(rng.normal(size=(8, 5)))
    user_vectors[0] = normalize_rows(np.array([[1, 1, 0, 0, 1]], dtype=float))[0]
    relevance = user_vectors @ item_vectors.T
    popularity = np.array([2000, 1500, 1200, 900, 700, 250, 180, 130, 100, 75, 60, 45, 35, 10, 8, 5, 3])
    clicks = np.maximum(1, np.round(popularity * np.clip(0.03 + 0.04 * np.maximum(item_vectors[:, 0], 0.0), 0.01, 0.20))).astype(int)
    exposure = np.clip(popularity / popularity.max(), 0.02, 1.0)
    rungs.append({
        "name": "D5 long-tail cold-start",
        "user_vectors": user_vectors,
        "item_vectors": item_vectors,
        "relevance": relevance,
        "positives": [np.argsort(-row)[:3] for row in relevance],
        "sessions": [[0, 1, 14], [1, 6, 15], [2, 8, 16], [3, 9, 13], [4, 10, 14], [5, 11, 15], [6, 12, 16], [7, 13, 14]],
        "targets": [14, 15, 16, 13, 14, 15, 16, 14],
        "clicks": clicks,
        "impressions": popularity.astype(int),
        "eligible": np.array([True] * 13 + [True, True, False, True]),
        "exposure": exposure,
        "cold_items": np.array([13, 14, 15, 16]),
    })

    return rungs


## The concept, built once: dot-product retrieval

A user tower $f_\theta(u)$ and an item tower $g_\phi(i)$ produce vectors, and retrieval scores them by

$$s(u,i)=f_\theta(u)^\top g_\phi(i).$$

The reusable method below implements dot or cosine scoring, brute-force top-k retrieval, sampled softmax, temperature, and recall@k.

In [ ]:

def two_tower_retrieve(user_vectors, item_vectors, positives, k=3, metric="dot", temperature=1.0):
    users = np.asarray(user_vectors, dtype=float)
    items = np.asarray(item_vectors, dtype=float)
    if metric == "cosine":
        users = normalize_rows(users)
        items = normalize_rows(items)
    scores = users @ items.T
    topk = np.argsort(-scores, axis=1)[:, :k]
    recall = recall_at_k(scores, positives, k=k)
    losses = []
    for row, positive_ids in zip(scores, positives):
        positive = int(np.atleast_1d(positive_ids)[0])
        probs = softmax(row, temperature=temperature)
        losses.append(-math.log(float(probs[positive]) + 1e-12))
    return {
        "scores": scores,
        "topk": topk,
        "recall": recall,
        "loss": float(np.mean(losses)),
    }


## Verify the lesson numbers once

With user $(1.0,0.2)$ and item vectors $(1,0.1)$, $(0.8,0.4)$, $(0.1,1)$, $(-0.5,0.8)$, the dot scores should be $1.020$, $0.880$, $0.300$, and $-0.340$. Sampled-softmax with scores $(2,1,0)$ should give positive probability $0.665$ and loss $0.408$; temperatures $0.5$, $1.0$, and $2.0$ should give $0.867$, $0.665$, and $0.506$.

In [ ]:

lesson_user = np.array([[1.0, 0.2]])
lesson_items = np.array([[1.0, 0.1], [0.8, 0.4], [0.1, 1.0], [-0.5, 0.8]])
lesson = two_tower_retrieve(lesson_user, lesson_items, [np.array([0])], k=3)
lesson_scores = lesson["scores"][0]
assert np.allclose(lesson_scores, [1.020, 0.880, 0.300, -0.340], atol=0.001)

sampled_scores = np.array([2.0, 1.0, 0.0])
probabilities = softmax(sampled_scores)
loss = -math.log(float(probabilities[0]))
assert round(float(probabilities[0]), 3) == 0.665
assert round(loss, 3) == 0.408

temperature_probs = [float(softmax(sampled_scores, temperature=t)[0]) for t in [0.5, 1.0, 2.0]]
assert [round(p, 3) for p in temperature_probs] == [0.867, 0.665, 0.506]
print("lesson scores", np.round(lesson_scores, 3))
print("temperature probabilities", np.round(temperature_probs, 3))


## The dataset ladder: D1 to D5

The same F14 recommender ladder is built inline: tiny slate, synthetic preferences, sparse noisy logs, a MovieLens-like genre matrix, and a long-tail cold-start catalog. The single metric here is **recall@3**.

In [ ]:

rungs = make_f14_ladder()
for rung in rungs:
    user_shape = rung["user_vectors"].shape
    item_shape = rung["item_vectors"].shape
    rel_shape = rung["relevance"].shape
    sparsity = float(np.mean(rung.get("exposure", np.ones(item_shape[0])) < 0.5))
    print(rung["name"], "users", user_shape, "items", item_shape, "relevance", rel_shape, "low exposure fraction", round(sparsity, 3))
    print("sample relevance row", np.round(rung["relevance"][0, : min(5, item_shape[0])], 3))


## Run the same method across D1-D5

Every rung uses the same two-tower dot-product retrieval function. D5 is intentionally long-tailed and includes cold items, so recall@3 exposes whether the retrieval stage still finds relevant tail candidates.

In [ ]:

results = []
for rung in rungs:
    output = two_tower_retrieve(rung["user_vectors"], rung["item_vectors"], rung["positives"], k=3)
    results.append({"name": rung["name"], "metric": output["recall"], "output": output})

for row in results:
    print(f"{row['name']:<28} recall@3={row['metric']:.3f}")


## Results visualization

The closing figure has one retrieved-neighbor panel per rung plus one recall@3 curve from D1 to D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()
for ax, rung, row in zip(flat_axes[:5], rungs, results):
    scores = row["output"]["scores"][0]
    order = np.argsort(-scores)[:5]
    ax.bar(np.arange(len(order)), scores[order], color="steelblue")
    ax.set_title(rung["name"])
    ax.set_xlabel("retrieved rank")
    ax.set_ylabel("dot score")
flat_axes[5].plot(range(1, 6), [row["metric"] for row in results], marker="o")
flat_axes[5].set_xticks(range(1, 6))
flat_axes[5].set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
flat_axes[5].set_ylim(0.0, 1.05)
flat_axes[5].set_title("recall@3 curve")
flat_axes[5].set_ylabel("recall@3")
fig.tight_layout()


## Pitfall on D5: easy negatives and unnormalized norms

Two towers cannot model ranker-level crosses before retrieval. On a long-tail D5 catalog, easy head negatives and raw dot products can hide hard cold items. The fix uses harder negatives through the full candidate set and explicit vector normalization.

In [ ]:

d5 = rungs[-1]
head_only_items = d5["item_vectors"][:8] * np.linspace(3.0, 1.2, 8)[:, None]
head_only_items = np.vstack([head_only_items, d5["item_vectors"][8:]])
wrong = two_tower_retrieve(d5["user_vectors"], head_only_items, d5["positives"], k=3, metric="dot")
fixed = two_tower_retrieve(d5["user_vectors"], d5["item_vectors"], d5["positives"], k=3, metric="cosine")
print("wrong raw-dot recall@3", round(wrong["recall"], 3))
print("fixed cosine hard-candidate recall@3", round(fixed["recall"], 3))
print("improvement", round(fixed["recall"] - wrong["recall"], 3))


## Evaluate it + practice

- Report **recall@3** against a no-skill popularity or random baseline on every rung.
- Sanity check that the D1 worked numbers match the lesson asserts before trusting larger rungs.
- Ablation: turn off vector normalization or restrict negatives to obvious head items; the metric should drop on D3-D5.
- Failure signals: head-item collapse, cold items never appearing, or a metric curve that improves only because labels are biased.
- Keep splits chronological or exposure-aware when the lesson's pitfall involves time or logging.

Practice: change the cutoff from 3 to 5 and compare the metric curve.

Practice: replace brute-force top-k with a two-stage candidate filter and check recall loss.